# Demo 03a — Linear Regression, Metrics, Polynomial Features and Regularization

**Class 03 · Block 1** (slides: `slides/03_linear_models.md`)

Five live demos on the **house prices** data (California housing: 20640 districts, 8 features, target = median house
value in 100 000 USD):

* **R1 — Data visualization, cleaning, preprocessing and splitting:** look at the data, clean it, fix its scale and
  skew, and split it without leaking.
* **R2 — The linear regression pipeline:** preprocessing + model fitted together; the normal equation, gradient
  descent with `jax.grad`, and scikit-learn.
* **R3 — Regression metrics and baselines:** MSE, RMSE, MAE, sMAPE and $R^2$ against the mean, median and mode.
* **R4 — Polynomial feature expansion:** still linear, and how it overfits.
* **R5 — Regularization:** ridge ($L_2$), lasso ($L_1$) and elastic net; choosing $\lambda$ by validation.

In [ ]:
from pathlib import Path

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet, Lasso, LassoCV, LinearRegression, Ridge, RidgeCV, lasso_path
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler

jax.config.update("jax_enable_x64", True)
plt.rcParams["figure.figsize"] = (9, 4)
DATA = Path("../../datasets")

## The data

The target is capped at 5.0 (500 000 USD) in the original data: a limit no linear model can see. Four features are
extremely skewed (a few districts have 1000+ people per house): `X_raw` keeps them as they are, `X` replaces them by
their logarithm (R1 shows why).

In [ ]:
housing = pl.read_csv(DATA / "california_housing.csv.zst")
names = [c for c in housing.columns if c != "MedHouseVal"]
skewed = [names.index(c) for c in ("AveRooms", "AveBedrms", "Population", "AveOccup")]
X_raw = housing.select(names).to_numpy().astype(float)
y = housing["MedHouseVal"].to_numpy().astype(float)
X = X_raw.copy()
X[:, skewed] = np.log(X[:, skewed])
print(housing.describe().select(["statistic", "MedInc", "HouseAge", "AveRooms", "Population", "MedHouseVal"]))
print("districts at the cap 5.0:", int((y >= 5.0).sum()), "of", len(y))

## R1 — Data visualization, cleaning, preprocessing and splitting

Four steps, in this order, before any model.

### 1. Data visualization

Look before you model: the **target**, the **features** and the **relations** between them. Every plot should end in a
decision (here: the cap at 5.0 is a limit for every model, a few features have extreme outliers, `MedInc` is the main
feature, and the coast matters).

In [ ]:
lat, lon = X[:, names.index("Latitude")], X[:, names.index("Longitude")]
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
sns.histplot(y, bins=40, ax=axes[0])
axes[0].set(xlabel="median house value (100 000 USD)", title=f"target: {int((y >= 5.0).sum())} districts at the cap 5.0")
sns.scatterplot(x=lon, y=lat, hue=y, palette="viridis", s=5, linewidth=0, ax=axes[1], legend=False)
axes[1].set(xlabel="longitude", ylabel="latitude", title="the coast is expensive (colour = value)")
sns.scatterplot(x=X[:, names.index("MedInc")], y=y, s=3, alpha=0.3, linewidth=0, ax=axes[2])
axes[2].set(xlabel="MedInc (10 000 USD)", ylabel="median value", title="income: the strongest single feature")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 4, figsize=(13, 3))
for ax, j in zip(axes, skewed):
    sns.histplot(np.clip(X_raw[:, j], None, np.percentile(X_raw[:, j], 99)), bins=40, ax=ax)
    ax.set(title=f"{names[j]} (raw, clipped at the 99th percentile)", xlabel="")
plt.tight_layout()
plt.show()
print("maximum / median of the skewed features:", {names[j]: round(float(X_raw[:, j].max() / np.median(X_raw[:, j])), 1) for j in skewed})

corr = np.corrcoef(np.c_[X, y].T)
plt.figure(figsize=(6.5, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0, xticklabels=[*names, "target"], yticklabels=[*names, "target"], cbar=False)
plt.title("correlations (four skewed features already log-transformed)")
plt.show()

### 2. Data cleaning

Check the data before trusting it: missing values, duplicated rows, impossible values, and values that are not what they
seem (here the **censored** target).

In [ ]:
print("missing values:", int(housing.null_count().to_numpy().sum()), "| duplicated rows:", int(housing.is_duplicated().sum()))
occ = X_raw[:, names.index("AveOccup")]
print("districts with more than 10 people per house:", int((occ > 10).sum()), "| with HouseAge at the cap 52:", int((X_raw[:, names.index("HouseAge")] >= 52).sum()))
print("districts with the target at the cap 5.0:", int((y >= 5.0).sum()), "(censored: the true value is above 5.0)")
# Dropping the censored rows changes the population we evaluate on: say so, or keep them and report the cap.
keep = y < 5.0
print(f"without the capped rows: {keep.sum()} districts, mean value {y[keep].mean():.3f} instead of {y.mean():.3f}")

**Missing values and categories** do not occur in this dataset, but they are the most common cleaning problem. Two real
datasets of the `datasets/` folder show the standard tools: a *median imputer* (with an indicator column) for numbers, and
a *one-hot encoder* for categories. Both **learn from the training rows only**.

In [ ]:
rng_c = np.random.default_rng(0)

# Missing numbers: auto_mpg has missing horsepower values.
auto = pl.read_csv(DATA / "auto_mpg.csv.zst")
Xa, ya = auto.drop("mpg").to_numpy().astype(float), auto["mpg"].to_numpy()
perm_a = rng_c.permutation(len(ya))
tr_a, te_a = perm_a[: int(0.8 * len(ya))], perm_a[int(0.8 * len(ya)) :]
print("auto_mpg missing values per column:", dict(zip(auto.drop("mpg").columns, auto.drop("mpg").null_count().row(0))))
imp = SimpleImputer(strategy="median", add_indicator=True).fit(Xa[tr_a])          # the median of the TRAINING rows
print("median horsepower learned on the training rows:", float(np.nanmedian(Xa[tr_a, 2])), "| on all the rows:", float(np.nanmedian(Xa[:, 2])))
Xa_tr = np.asarray(imp.transform(Xa[tr_a]))
print("columns after imputation (the last ones flag the imputed columns):", Xa_tr.shape[1], "| missing values left:", int(np.isnan(Xa_tr).sum()))

# Categories: heart_disease has categorical columns (and a few missing values).
heart = pl.read_csv(DATA / "heart_disease.csv.zst")
cat_cols = [c for c, t in zip(heart.columns, heart.dtypes) if t == pl.String and c != "num"]
num_cols = [c for c in heart.columns if c not in cat_cols and c != "num"]
perm_h = rng_c.permutation(heart.height)
tr_k, te_k = perm_h[: int(0.8 * heart.height)], perm_h[int(0.8 * heart.height) :]
Xc = heart.select(cat_cols).with_columns(pl.all().fill_null("missing")).to_numpy()      # a missing category is a category
enc = OneHotEncoder(handle_unknown="ignore").fit(Xc[tr_k])                               # the levels of the TRAINING rows
Xc_tr = np.asarray(enc.transform(Xc[tr_k]).toarray())  # pyright: ignore[reportAttributeAccessIssue]
print(f"heart_disease: {len(cat_cols)} categorical columns -> {Xc_tr.shape[1]} one-hot columns, e.g. {list(enc.categories_[1])}")
print("numeric columns with missing values:", {c: int(heart[c].null_count()) for c in num_cols if heart[c].null_count()})

### 3. Preprocessing

Whatever is learned from the data is fitted on the **training part only**. Two transformations matter here: the
**logarithm** of the skewed features and the **standardization** of every feature.

In [ ]:
perm0 = np.random.default_rng(1).permutation(len(y))
te0, tr0 = perm0[: len(y) // 5], perm0[len(y) // 5 :]

# Why a logarithm? Least squares on the raw skewed features against the logged ones (the test MSE is scale-free).
for label, M in (("raw features", X_raw), ("log of the skewed ones", X)):
    m = LinearRegression().fit(M[tr0], y[tr0])
    pr = m.predict(M[te0])
    print(f"{label:<24} test MSE {mean_squared_error(y[te0], pr):.3f}  R2 {r2_score(y[te0], pr):.3f}")

# Why standardize? The condition number of the loss Hessian (Class 02) is the speed limit of gradient descent.
for label, M in (("raw units", X[tr0]), ("standardized", StandardScaler().fit(X[tr0]).transform(X[tr0]))):
    A = np.c_[M, np.ones(len(M))]
    ev = np.linalg.eigvalsh(2 * A.T @ A / len(A))
    print(f"{label:<24} condition number {ev.max() / ev.min():.3g}")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
sns.boxplot(data=X[tr0], orient="h", fliersize=1, ax=axes[0])
axes[0].set(yticks=range(8), yticklabels=names, title="log features, raw units: very different scales")
sns.boxplot(data=StandardScaler().fit(X[tr0]).transform(X[tr0]), orient="h", fliersize=1, ax=axes[1])
axes[1].set(title="standardized with the training statistics")
plt.tight_layout()
plt.show()

### 4. Splitting

Every measurement of a model must come from data it has **not** trained on. Three ways to organize that:

1. **Hold-out:** one split (80% train, 20% test). Cheap, but one number.
2. **Repeated hold-out:** many random splits: shows how much one split can mislead.
3. **k-fold cross-validation:** every example is tested exactly once.

The model is always the same linear regression, with the features standardized **using the training part only**.

In [ ]:
def test_mse(train_idx, test_idx):
    scaler = StandardScaler().fit(X[train_idx])                      # statistics of the training part only
    model = LinearRegression().fit(scaler.transform(X[train_idx]), y[train_idx])
    return mean_squared_error(y[test_idx], model.predict(scaler.transform(X[test_idx])))


rng = np.random.default_rng(0)
single = []
for _ in range(200):
    perm = rng.permutation(len(y))
    single.append(test_mse(perm[len(y) // 5 :], perm[: len(y) // 5]))
single = np.array(single)
print(f"one 80/20 split, repeated 200 times: test MSE from {single.min():.3f} to {single.max():.3f} (mean {single.mean():.3f}, std {single.std():.3f})")

for k in (5, 10):
    scores = [test_mse(a, b) for a, b in KFold(k, shuffle=True, random_state=0).split(X)]
    half = 2.26 * np.std(scores, ddof=1) / np.sqrt(k) if k == 10 else 2.78 * np.std(scores, ddof=1) / np.sqrt(k)
    print(f"{k:2d}-fold CV: mean {np.mean(scores):.3f}, std of folds {np.std(scores, ddof=1):.3f}, 95% CI half-width {half:.3f}")

plt.hist(single, bins=25, color="tab:blue", alpha=0.8)
plt.axvline(single.mean(), color="k", ls="--")
plt.xlabel("test MSE of one random 80/20 split")
plt.title("the same model, 200 different splits")
plt.show()

With 4128 test districts the split-to-split spread is small; with a few hundred examples it would be large. The
folds of a cross-validation give a mean **and** a spread from a single experiment (Class 01: confidence intervals).

**Order matters.** If the file is sorted (by date, by region), a split by position tests on a *different world*.
Shuffle first, unless the data is a time series: then always train on the past and test on the future.

## R2 — The linear regression pipeline

$$\hat y = w^\top x + b, \qquad \mathcal{L}(w, b) = \frac1N\sum_i (\hat y_i - y_i)^2$$

The model is a **pipeline**: standardization (fitted on the training part) followed by the regression. We fix one 80/20
split and build the same model three ways: scikit-learn's `Pipeline`, the normal equation and gradient descent.

In [ ]:
perm = np.random.default_rng(1).permutation(len(y))
te, tr = perm[: len(y) // 5], perm[len(y) // 5 :]
scaler = StandardScaler().fit(X[tr])
Xtr, Xte = np.asarray(scaler.transform(X[tr])), np.asarray(scaler.transform(X[te]))
ytr, yte = y[tr], y[te]

# 1. The normal equation: set the gradient to zero, solve a linear system.
A = np.c_[Xtr, np.ones(len(Xtr))]
theta_ne = np.linalg.solve(A.T @ A, A.T @ ytr)

# 2. Gradient descent, with the gradient from jax.grad (no derivative written by hand).
def loss(theta, X_, y_):
    return jnp.mean((X_ @ theta[:-1] + theta[-1] - y_) ** 2)

grad = jax.jit(jax.grad(loss))

def gradient_descent(lr, steps):
    theta = jnp.zeros(Xtr.shape[1] + 1)
    history = [float(loss(theta, Xtr, ytr))]
    for _ in range(steps):
        theta = theta - lr * grad(theta, Xtr, ytr)
        history.append(float(loss(theta, Xtr, ytr)))
    return theta, np.array(history)

theta_gd, hist = gradient_descent(0.4, 300)          # lr just below 2 / lambda_max (see below)

# 3. scikit-learn.
sk = LinearRegression().fit(Xtr, ytr)

print("max |normal equation - sklearn| =", np.abs(theta_ne - np.r_[sk.coef_, sk.intercept_]).max())
print("max |gradient descent - sklearn| =", np.abs(np.asarray(theta_gd) - np.r_[sk.coef_, sk.intercept_]).max())
print(pl.DataFrame({"feature": names + ["intercept"], "coefficient": np.round(theta_ne, 3)}))

# 4. The whole pipeline in one object: it learns the scaling and the weights from the training part only.
pipe = make_pipeline(StandardScaler(), LinearRegression()).fit(X[tr], ytr)
print("max |pipeline - normal equation| on the test predictions =", np.abs(pipe.predict(X[te]) - np.c_[Xte, np.ones(len(Xte))] @ theta_ne).max())

The coefficients are in **standardized** units: the effect on the price (in 100 000 USD) of one standard deviation
of the feature. Latitude and longitude carry large negative weights (the coast is expensive), income a large
positive one.

**The learning rate comes from the curvature** (Class 02). The Hessian of the MSE is constant, $H = \frac2N A^\top A$,
and gradient descent is stable only for $\eta < 2/\lambda_{\max}(H)$.

In [ ]:
H = np.asarray(jax.hessian(loss)(jnp.zeros(9), Xtr, ytr))
eig = np.linalg.eigvalsh(H)
print(f"Hessian eigenvalues: min {eig.min():.3f}, max {eig.max():.3f} -> condition number {eig.max() / eig.min():.1f}; stable for lr < {2 / eig.max():.3f}")

fig, axes = plt.subplots(1, 2)
optimum = float(loss(jnp.asarray(theta_ne), Xtr, ytr))
for lr in (0.02, 0.1, 0.4, 2 / eig.max() * 1.02):
    _, h = gradient_descent(lr, 100)
    axes[0].plot(np.maximum(h - optimum, 1e-16), label=f"lr = {lr:.2f}")
axes[0].set(yscale="log", ylim=(1e-12, 1e4), xlabel="step", ylabel="loss - optimum (log)")
axes[0].legend()
pred = np.c_[Xte, np.ones(len(Xte))] @ theta_ne
axes[1].scatter(yte, pred, s=3, alpha=0.3)
axes[1].plot([0, 5], [0, 5], "k--")
axes[1].set(xlabel="actual value", ylabel="predicted value", title="test set: the cap at 5.0 is visible")
plt.tight_layout()
plt.show()

Closed form or gradient descent? The normal equation solves a $d\times d$ system, $O(Nd^2 + d^3)$: perfect for 8
features and hopeless for a million. Gradient descent costs $O(Nd)$ per step and works for **any** differentiable
loss. Linear regression is the one model where we can check the iterative answer against the exact one.

## R3 — Regression metrics and baselines

$$\text{MSE} = \tfrac1n\sum e_i^2,\quad \text{RMSE} = \sqrt{\text{MSE}},\quad
\text{MAE} = \tfrac1n\sum |e_i|,\quad
\text{sMAPE} = \tfrac{100}{n}\sum \tfrac{2|e_i|}{|y_i| + |\hat y_i|},\quad
R^2 = 1 - \tfrac{\sum e_i^2}{\sum (y_i - \bar y)^2}$$

In [ ]:
def report(y_true, y_pred):
    e = y_true - y_pred
    return {
        "MSE": np.mean(e**2),
        "RMSE": np.sqrt(np.mean(e**2)),
        "MAE": np.mean(np.abs(e)),
        "sMAPE %": 100 * np.mean(2 * np.abs(e) / (np.abs(y_true) + np.abs(y_pred))),
        "R2": 1 - np.sum(e**2) / np.sum((y_true - y_true.mean()) ** 2),
    }


rows = {"mean baseline": report(yte, np.full(len(yte), ytr.mean())), "linear regression": report(yte, pred)}
print(pl.DataFrame({"model": list(rows)} | {k: [round(float(r[k]), 3) for r in rows.values()] for k in rows["mean baseline"]}))
print("check with sklearn: R2", round(r2_score(yte, pred), 3), "MAE", round(mean_absolute_error(yte, pred), 3))

**Statistical baselines.** The simplest model answers with a constant: the **mean**, the **median** or the **mode** of the
training target. Which constant is best depends on the metric: the mean minimizes the squared error, the median the
absolute error. A model must beat the best baseline *for the metric we report*.

In [ ]:
vals, counts = np.unique(np.round(ytr, 2), return_counts=True)
constants = {"mean": float(ytr.mean()), "median": float(np.median(ytr)), "mode": float(vals[counts.argmax()])}
rows = {f"{k} ({v:.3f})": report(yte, np.full(len(yte), v)) for k, v in constants.items()} | {"linear regression": report(yte, pred)}
print(pl.DataFrame({"model": list(rows)} | {k: [round(float(r[k]), 3) for r in rows.values()] for k in report(yte, pred)}))

fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
cs = np.linspace(0.2, 5.2, 300)
axes[0].plot(cs, [np.mean((yte - c) ** 2) for c in cs], label="MSE")
axes[0].plot(cs, [np.mean(np.abs(yte - c)) for c in cs], label="MAE")
for (k, c), color in zip(constants.items(), ("tab:green", "tab:red", "tab:purple")):
    axes[0].axvline(c, color=color, ls="--", label=f"{k} {c:.2f}")
axes[0].set(xlabel="constant answer", ylabel="test error", title="the best constant depends on the metric")
axes[0].legend(fontsize=8)
labels = list(rows)
x = np.arange(len(labels))
axes[1].bar(x - 0.2, [rows[k]["RMSE"] for k in labels], 0.4, label="RMSE")
axes[1].bar(x + 0.2, [rows[k]["MAE"] for k in labels], 0.4, label="MAE")
axes[1].set(xticks=x, xticklabels=[k.split(" (")[0] for k in labels], title="error in units of 100 000 USD")
axes[1].legend()
axes[2].bar(x, [rows[k]["R2"] for k in labels], color="tab:gray")
axes[2].set(xticks=x, xticklabels=[k.split(" (")[0] for k in labels], title="$R^2$ (0 = the mean baseline)")
axes[2].axhline(0, color="black", lw=0.8)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 3.4))
sns.histplot({"mean baseline": yte - constants["mean"], "linear regression": yte - pred}, bins=50, element="step", fill=False, ax=ax)
ax.set(xlabel="residual (actual - predicted)", title="the model shrinks and centres the errors")
plt.show()

The mean wins the **MSE** (1.32) and the median wins the **MAE** (0.875 against 0.904): each constant is optimal for its own
loss. The **mode** is the most frequent value, which here is the cap at 5.0: a terrible constant for a continuous target
($R^2 \ll 0$). The linear regression beats all three on every metric, which is what makes it worth reporting.

**What each metric hides.** Five toy experiments:

* **Outliers:** one huge error dominates MSE (squared) but barely moves MAE.
* **sMAPE is asymmetric:** the same absolute error costs more when we *under*-predict.
* **$R^2$ can be negative:** worse than predicting the mean.

In [ ]:
y_true = np.array([2.0, 2.5, 3.0, 3.5, 4.0])
good = y_true + np.array([0.1, -0.1, 0.1, -0.1, 0.1])
outlier = good.copy()
outlier[2] += 3.0                                    # one prediction is 3 units off
print("small errors     :", {k: round(float(v), 3) for k, v in report(y_true, good).items()})
print("plus one outlier :", {k: round(float(v), 3) for k, v in report(y_true, outlier).items()})

print("\nsMAPE asymmetry, true value 100:")
for p in (50, 150):
    print(f"  prediction {p}: error 50 -> sMAPE {100 * 2 * 50 / (100 + p):.1f}%   plain percentage error {100 * 50 / 100:.0f}%")

bad = np.full(5, 10.0)
print("\nR2 of a constant prediction of 10:", round(float(report(y_true, bad)["R2"]), 2))

**Which to report?** MSE/RMSE if large errors are especially costly (RMSE is in the unit of $y$); MAE if you want a
typical error robust to outliers; sMAPE for a relative error across scales (never near 0); $R^2$ to compare with the
mean baseline, but never alone.

## R4 — Polynomial feature expansion

A model is *linear* when it is linear in the **parameters**. Adding the columns $x_j x_k$, $x_j^2$, ... to $X$ gives
a curved model that the same least-squares machinery fits. For $n$ features and degree $d$ there are
$\binom{n+d}{d} - 1$ new columns: for the 8 house features, degrees 1 to 4 give 8, 44, 164 and 494.

In [ ]:
rng = np.random.default_rng(3)
idx = rng.choice(len(y), 25, replace=False)
x1 = (X[idx, 0] - X[:, 0].mean()) / X[:, 0].std()                  # median income, standardized
grid = np.linspace(x1.min() - 0.2, x1.max() + 0.2, 300)

fig, axes = plt.subplots(1, 4, figsize=(14, 3.2), sharey=True)
for ax, d in zip(axes, (1, 3, 9, 15)):
    coef = np.polyfit(x1, y[idx], d)
    ax.scatter(x1, y[idx], s=10, color="k")
    ax.plot(grid, np.polyval(coef, grid), color="tab:red")
    ax.set(title=f"degree {d}", ylim=(0, 6))
plt.show()

In [ ]:
sub = np.random.default_rng(4).choice(tr, 300, replace=False)
print("features for degrees 1-4:", [PolynomialFeatures(d, include_bias=False).fit(X[:2]).n_output_features_ for d in range(1, 5)])

def degree_sweep(n_train, degrees=(1, 2, 3, 4)):
    chosen = np.random.default_rng(4).choice(tr, n_train, replace=False)
    out = []
    for d in degrees:
        poly = PolynomialFeatures(d, include_bias=False).fit(X[chosen])
        P, Pt = np.asarray(poly.transform(X[chosen])), np.asarray(poly.transform(X[te]))
        sc = StandardScaler().fit(P)
        m = LinearRegression().fit(sc.transform(P), y[chosen])
        out.append((d, P.shape[1], mean_squared_error(y[chosen], m.predict(sc.transform(P))), mean_squared_error(y[te], m.predict(sc.transform(Pt)))))
    return out

for n in (300, 3000):
    print(f"\n{n} training examples")
    for d, p, a, b in degree_sweep(n):
        print(f"  degree {d}: {p:4d} features  train MSE {a:9.4f}  test MSE {b:12.4f}")

With 300 examples the training error keeps falling (at degree 4 there are more features than examples, so the model
interpolates) while the test error explodes. With 3000 examples the explosion is much smaller (degree 3: test MSE 1.0 instead of 160) and the flexible models
stop being absurd, but the plain line is still as good: **more data tames a flexible model**, it does not make flexibility pay by itself (Class 01: bias, variance and the learning curve). The other cure is to constrain the model: regularization.

## R5 — Regularization

$$\min_{w,b}\ \frac1N\sum_i (\hat y_i - y_i)^2 + \lambda\,\Omega(w), \qquad
\Omega_{\text{ridge}} = \lVert w\rVert_2^2 \;(L_2),\quad \Omega_{\text{lasso}} = \lVert w\rVert_1 \;(L_1)$$

The penalty is a *description length* for the weights (Class 01: MDL) and, equivalently, a Gaussian (ridge) or Laplace
(lasso) **prior** on them. Elastic net mixes both. The bias is never penalized.

**Geometry.** The least-squares contours meet the constraint region $\lVert w\rVert_p \le t$ where the loss is smallest
*inside* it. The $L_2$ ball is round, so the contact point has no zero coordinate; the $L_1$ ball has **corners on the
axes**, and the contact is often a corner: a weight is exactly 0.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
w1, w2 = np.meshgrid(np.linspace(-1, 3, 300), np.linspace(-1, 3, 300))
loss2d = 4 * (w1 - 2.0) ** 2 + (w2 - 1.6) ** 2 + 2.4 * (w1 - 2.0) * (w2 - 1.6)
theta_ = np.linspace(0, 2 * np.pi, 400)
for ax, kind in zip(axes, ("L2 (ridge)", "L1 (lasso)")):
    ax.contour(w1, w2, loss2d, levels=8, colors="tab:blue", linewidths=0.8)
    if kind.startswith("L2"):
        r = 1.5
        ax.plot(r * np.cos(theta_), r * np.sin(theta_), "k", lw=2)
    else:
        t = 1.5
        ax.plot([t, 0, -t, 0, t], [0, t, 0, -t, 0], "k", lw=2)
    ax.plot(2.0, 1.6, "b*", ms=12)
    ax.set(title=kind, xlabel="$w_1$", ylabel="$w_2$", aspect="equal", xlim=(-1, 3), ylim=(-1, 3))
plt.show()

In [ ]:
# 300 examples, degree-3 features (164 columns): plain least squares explodes; regularization tames it.
poly = PolynomialFeatures(3, include_bias=False).fit(X[sub])
sc = StandardScaler().fit(poly.transform(X[sub]))
P, Pt = sc.transform(poly.transform(X[sub])), sc.transform(poly.transform(X[te]))
ys = y[sub]
plain = LinearRegression().fit(P, ys)
print("plain least squares, test MSE:", round(mean_squared_error(yte, plain.predict(Pt)), 2))

alphas = np.logspace(-2, 4, 25)              # sklearn's Ridge alpha = lambda * N
ridge_test, ridge_train = [], []
for a in alphas:
    m = Ridge(alpha=a).fit(P, ys)
    ridge_test.append(mean_squared_error(yte, m.predict(Pt)))
    ridge_train.append(mean_squared_error(ys, m.predict(P)))
best = alphas[int(np.argmin(ridge_test))]

fig, axes = plt.subplots(1, 2)
axes[0].plot(alphas, ridge_train, label="train")
axes[0].plot(alphas, ridge_test, label="test")
axes[0].axvline(best, color="gray", ls="--")
axes[0].set(xscale="log", yscale="log", xlabel="ridge (L2) alpha (log, stronger to the right)", ylabel="MSE (log)")
axes[0].legend()
axes[1].plot(alphas, np.array([Ridge(alpha=a).fit(P, ys).coef_ for a in alphas]), lw=0.6)
axes[1].set(xscale="log", xlabel="ridge (L2) alpha (log)", ylabel="coefficients", title="ridge (L2) path: smooth shrinkage")
plt.tight_layout()
plt.show()
print(f"best ridge alpha {best:.3g}: test MSE {min(ridge_test):.3f} (plain: {mean_squared_error(yte, plain.predict(Pt)):.1f})")

In [ ]:
alphas_l, coefs_l, _ = lasso_path(P, ys, alphas=np.logspace(-3, 0, 40), max_iter=100_000)  # pyright: ignore[reportArgumentType]
n_nonzero = (coefs_l != 0).sum(axis=0)
lasso_test = [mean_squared_error(yte, Lasso(alpha=a, max_iter=100_000).fit(P, ys).predict(Pt)) for a in alphas_l]
best_l = alphas_l[int(np.argmin(lasso_test))]

fig, axes = plt.subplots(1, 2)
axes[0].plot(alphas_l, coefs_l.T, lw=0.6)
axes[0].set(xscale="log", xlabel="lasso (L1) alpha (log)", ylabel="coefficients", title="lasso (L1) path: weights hit exactly 0")
axes[1].plot(alphas_l, n_nonzero, color="tab:red")
axes[1].set(xscale="log", xlabel="lasso (L1) alpha (log)", ylabel="non-zero coefficients (of 164)")
plt.tight_layout()
plt.show()
i = int(np.argmin(lasso_test))
print(f"best lasso alpha {best_l:.3g}: test MSE {lasso_test[i]:.3f} with {n_nonzero[i]} of 164 features")

en = ElasticNet(alpha=best_l, l1_ratio=0.5, max_iter=100_000).fit(P, ys)
print(f"elastic net (l1_ratio 0.5, alpha {best_l:.3g}): test MSE {mean_squared_error(yte, en.predict(Pt)):.3f}, non-zero {int((en.coef_ != 0).sum())}")

**Choosing $\lambda$.** The test set above was used to *draw* the curves, which is fine for a demo but not for a
decision: choose $\lambda$ on a **validation** set or by cross-validation (`RidgeCV`, `LassoCV`) and touch the test set
once, at the end.

In [ ]:
ridge_cv = RidgeCV(alphas=np.logspace(-2, 4, 25), cv=5).fit(P, ys)
lasso_cv = LassoCV(alphas=np.logspace(-3, 0, 40), cv=5, max_iter=100_000).fit(P, ys)  # pyright: ignore[reportArgumentType]
print(f"RidgeCV chose alpha {ridge_cv.alpha_:.3g}: test MSE {mean_squared_error(yte, ridge_cv.predict(Pt)):.3f}")
print(f"LassoCV chose alpha {lasso_cv.alpha_:.3g}: test MSE {mean_squared_error(yte, lasso_cv.predict(Pt)):.3f}, non-zero {int((lasso_cv.coef_ != 0).sum())}")

## Summary

* Linear regression = least squares: the normal equation is exact, gradient descent (`jax.grad`) is general.
* Report several metrics and always a baseline; $R^2 = 0$ *is* the mean baseline.
* Polynomial features make the model curved and flexible; without enough data it overfits.
* Regularization shrinks the weights (ridge) or removes some (lasso); $\lambda$ is a hyperparameter chosen by
  validation.